# EDA — fraude par carte bancaire

Notebook reproductible fondé sur le module `src/analysis.py`. Le CSV brut reste local et ignoré par Git. Installez les dépendances depuis `requirements.txt`. Si seul le jeu synthétique est disponible, les résultats sont uniquement une démonstration.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import seaborn as sns
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.analysis import load_csv, quality_report, summary, univariate_report, class_comparison, hourly_profile, sha256_file
path = ROOT / 'data' / 'creditcard.csv'
if not path.exists():
    raise FileNotFoundError('Ajoutez le CSV réel ignoré dans data/creditcard.csv. Utilisez app.py pour découvrir la démo synthétique.')
print('SHA-256:', sha256_file(path))
df = load_csv(path)
df.shape

## Qualité et structure

Les valeurs manquantes et doublons sont quantifiés. Aucun enregistrement n'est supprimé sans justification.

In [ ]:
quality_report(df), df.dtypes.astype(str).to_dict()

## Déséquilibre et variables univariées

In [ ]:
summary(df), univariate_report(df).sort_values('skewness', key=lambda x: x.abs(), ascending=False).head(12)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=df, x='Amount', hue='Class', bins=60, element='step', stat='density', common_norm=False, ax=axes[0])
sns.histplot(data=df, x='Time', hue='Class', bins=60, element='step', stat='density', common_norm=False, ax=axes[1])
plt.tight_layout()

## Comparaison fraude / non-fraude

Mann–Whitney et KS s'accompagnent respectivement d'un effet rank-bisérial et d'une distance KS. Les p-values corrigées BH tiennent compte des comparaisons multiples.

In [ ]:
comparisons = class_comparison(df)
comparisons.head(15)

In [ ]:
top = comparisons.head(12).sort_values('ks_statistic')
sns.barplot(data=top, x='ks_statistic', y='variable', color='#11865D')
plt.xlabel('Distance KS'); plt.ylabel('Variable'); plt.tight_layout()

## Temps relatif et limites

`Time` est relatif au début de la collecte. Son modulo 24 h est une approximation cyclique, pas une heure civile. Les différences observées ne démontrent aucune causalité. `V1`–`V28` sont des composantes PCA anonymisées. Aucun modèle n'est entraîné.

In [ ]:
hourly_profile(df)